In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import textwrap
from IPython.display import display, Markdown
from PIL import Image
from wordcloud import WordCloud, STOPWORDS
from nltk.corpus import stopwords
from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import LearningCurveDisplay, ShuffleSplit
from sklearn.model_selection import validation_curve
from fitter import Fitter, get_common_distributions
from scipy.stats import zscore
import math
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import warnings
from scipy.optimize import curve_fit

# %run ./Data_Visualizations.ipynb

warnings.filterwarnings('ignore')

seed=1123

In [ ]:
class color_scheme:

    def pmi_colors():

        colorlist = ['#000000', '#05bfe0', "#207ace", "#ff610f", '#fc630c', '#4f17a8', '#4e16a4',
                         '#040404', '#fc6414', '#40643c', '#44043c']
        
        color_scheme = {"colorlist": colorlist, 
                        "colormap": LinearSegmentedColormap.from_list("my_custom_colormap", colorlist), 
                        "backgrounds": {'dark': 'white',
                                        'light': '#000000',}
                       }

        plt.rcParams['font.family'] = 'Arial'
        plt.rcParams['axes.grid'] = True
        plt.rcParams['grid.color'] = 'gray'
        plt.rcParams['grid.linewidth'] = 0.5
        plt.rcParams['grid.linestyle'] = '--'
        plt.rcParams['grid.alpha'] = 0.7
        plt.rcParams["axes.axisbelow"] = True
        plt.rcParams['axes.spines.top'] = False
        plt.rcParams['axes.spines.right'] = False
        plt.rcParams['axes.spines.bottom'] = False
        plt.rcParams['axes.spines.left'] = False
        
        return color_scheme

    def bpn_colors():

        colorlist = ['#ff4d63', '#ffb9c1', '#2d8bba',  '#41b8d5', '#6ce5e8']
        
        color_scheme = {"colorlist": colorlist, 
                        "colormap": LinearSegmentedColormap.from_list("my_custom_colormap", colorlist), 
                        "backgrounds": {'dark': 'white',
                                        'light': '#000000'}
                       }

        plt.rcParams['font.family'] = 'Century Gothic'
        plt.rcParams['axes.grid'] = True
        plt.rcParams['grid.color'] = 'lightgray'
        plt.rcParams['grid.linewidth'] = 0.5
        # plt.rcParams['grid.linestyle'] = '--'
        plt.rcParams['grid.alpha'] = 0.7
        plt.rcParams["axes.axisbelow"] = True
        plt.rcParams["font.size"] = 14
        plt.rcParams['axes.spines.top'] = False
        plt.rcParams['axes.spines.right'] = False
        plt.rcParams['axes.spines.bottom'] = False
        plt.rcParams['axes.spines.left'] = False
        
        return color_scheme

In [1]:
class plot:

    def sprint_velocity(x_col, y_cols, df, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']

        # Plot the DataFrame, specifying the x-axis and kind='bar'
        df.plot(x=x_col, y=y_cols, kind="bar", color = colorlist[1:3])
        df['Average'] = df[y_cols[-1]].rolling(window=2).mean()
        df['Average'].plot(kind='line', color=colorlist[0], marker='o')
        
        # Add title and rotate labels if needed
        plt.title('Sprint Velocity')
        plt.xticks(rotation=0) # Keeps x-labels horizontal
        plt.ylabel('Story Points')
        plt.legend(loc='center left',
                  bbox_to_anchor=(1, 0.9),
                  fontsize=10)
        plt.show()

    def stakeholder_engagement(current, desired, names, scheme=None):
        """current and desired should be ints on a scale of 1 to 5"""
        
        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']

        fig, ax = plt.subplots(figsize=(3, 3))
        
        for ix, x in enumerate(current):
            line = [current[ix], desired[ix]]
            
            plt.plot(['Current', 'Desired'],
                     line,
                     marker='o',
                     color = colorlist[ix])

        # Force integer ticks on the y-axis
        ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        labels = [item.get_text() for item in ax.get_yticklabels()]
        labels[1] = 'Unaware'
        labels[2] = 'Resistant'
        labels[3] = 'Neutral'
        labels[4] = 'Supporting'
        labels[5] = 'Leading'
        
        ax.set_yticklabels(labels)
        ax.legend(labels=names,
                  loc='center left',
                  bbox_to_anchor=(1.05, 0.75),
                  fontsize=10)
        plt.title('Stakeholder Engagement Assessment')
        plt.show()
    
    def power_interest(impact, influence, names, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        
        fig, ax = plt.subplots()

        # Loop through each data point and add text
        for i in range(len(names)):
        
            if impact[i]>=5:
                if influence[i]>=5:
                    color = colorlist[0]
                else:
                    color = colorlist[1]
            else:
                if influence[i]>=5:
                    color = colorlist[2]
                else:
                    color = colorlist[3]
        
            name_split = names[i].split()
            max_split = max(len(name_split[0]), len(name_split[1]))
            
            plt.axvline(5, color='gray', linestyle="dotted")
            plt.axhline(5, color='gray', linestyle="dotted")
            
            plt.text(
                x=impact[i], 
                y=influence[i], 
                s='\n'.join(textwrap.wrap(names[i], max_split)),
                ha='center', # Horizontal alignment
                va='center', # Vertical alignment
                fontsize=10,
                color=color
            )

        # Set axis labels and title
        plt.xlabel('Stakeholder Impact')
        plt.ylabel('Stakeholder Influence')
        plt.title('Power-Interest Grid')
        
        handles = [
            mpatches.Patch(color=colorlist[0], label='Collaborate With'),
            mpatches.Patch(color=colorlist[1], label='Keep Informed'),
            mpatches.Patch(color=colorlist[2], label='Maintain Confidence'),
            mpatches.Patch(color=colorlist[3], label='Monitor')
        ]

        ax.legend(handles=handles,
                  loc='center left',
                  bbox_to_anchor=(1.05, 0.85),
                  fontsize=10)
        
        # Adjust plot limits for better visibility
        plt.xlim(0, 10.5)
        plt.ylim(0, 10.5)
        
        # Display the plot
        plt.show()
    
    def usage(x, y, chart_type=None, title=None, label=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            if mode=='dark':
                continue
            
            fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed

            if chart_type=='line':
                plt.plot(x, y[0], color=colorlist[0], label=label[0])
                plt.plot(x, y[1], color=colorlist[3], label=label[1], linestyle='dashed')
                plt.ylabel('Submissions (#)', color=color)
                ax.tick_params(axis='x', rotation=45, labelcolor=color, colors=color)
                ax.legend(bbox_to_anchor=(1, 0.975),
                          loc='upper left',
                          labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            elif chart_type=='bar':
                plt.bar(x, y, color=colorlist[0])
                plt.ylabel('Submissions (#)', color=color)
                ax.tick_params(axis='x', rotation=45, labelcolor=color, colors=color)
            elif chart_type=='barh':
                plt.barh(x, y, color=colorlist[0])
                plt.xlabel('Submissions (#)', color=color)
                plt.gca().invert_yaxis()
                
            plt.title(title, color=color)
            
            ax.tick_params(axis='both', labelcolor=color, colors=color)

            # # handles = ['Last week']
            # ax.legend(handles=list(set(handles)),
            #           bbox_to_anchor=(1, 0.975),
            #           labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color

            plot_filename = "Images\\"+mode+"_line_plot_"+title.replace(" ", "_").lower()+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def line(x, y, title=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed
            
            index = y.idxmax()
            plt.text(x[index], y.max(), "*", color=color)
            plt.text(x[index]-6.5, y.max()+.015, "Seed: " + str(x[index]), color=color)
            
            plt.plot(x, y, color=colorlist[0])
            plt.ylim(y.min()-.15 , y.max()+.15)
            
            plt.title(title, color=color)
            ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
            ax.tick_params(axis='both', labelcolor=color, colors=color) 

            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color

            plot_filename = "Images\\"+mode+"_line_plot_"+title.replace(" ", "_").lower()+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)
            
    def bar_100_stacked(df, top_group, group, orientation=None, title=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        top_groups = df.groupby(top_group)[top_group].count()
    
        group_1 = top_groups.index[0]
        group_2 = top_groups.index[1]
        
        grouped = df.groupby([top_group, group])[group].count()
    
        group_1_grouped = grouped[group_1].sort_index(ascending=True)
        group_2_grouped = grouped[group_2].sort_index(ascending=True)
    
        results = {group_1: [],
                   group_2: []}

        category_list = list(df.groupby(group)[group].count().index)
    
        for category in category_list:
        
            try:
                num1 = group_1_grouped.loc[category]/group_1_grouped.sum()*100
            except:
                num1 = 0
        
            results[group_1].append(num1)
        
            try:
                num2 = group_2_grouped.loc[category]/group_2_grouped.sum()*100
            except:
                num2 = 0
    
            results[group_2].append(num2)
    
        labels = list(results.keys())
        data = np.array(list(results.values()))
        data_cum = data.cumsum(axis=1)

        fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed

        # Set up colors (optional, but recommended for clarity)
        category_colors = colormap(np.linspace(0.15, 0.85, data.shape[1]))
        
        for i, col_name in enumerate(category_list):
            widths = data[:, i]
            starts = data_cum[:, i] - widths
            if orientation=='horizontal':
                ax.barh(labels, widths, left=starts, height=0.6, label=col_name, alpha=0.5, color=category_colors[i])
                ax.xaxis.set_major_formatter(FuncFormatter(percentage_formatter))
            elif orientation=='vertical':
                ax.bar(labels, widths, bottom=starts, width=0.8, label=col_name, alpha=0.5, color=category_colors[i])
                ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
    
        for mode, color in backgrounds.items():
            
            COLOR = color
        
            plt.title(title, color=COLOR)
            ax.tick_params(axis='both', labelcolor=COLOR, colors=COLOR) 

            ax.legend(bbox_to_anchor=(1, 0.975),
                      loc='upper left',
                      labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)

            plot_filename = "Images\\"+mode+"_"+orientation+"_100_stacked_barplot_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def bar(df, group, orientation=None, title=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        groups = df.groupby(group)[group].count()

        sorted_groups = groups.sort_values(ascending=True)
        labels = list(sorted_groups.index)
        data = sorted_groups.values

        fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed

        label_locs = np.arange(len(labels))  # the label locations

        for mode, color in backgrounds.items():
            COLOR = color

            if orientation=='horizontal':
                column_width = 1
                # the width of the bars
                rects = ax.barh(label_locs, data, alpha=0.5, color=colorlist[0]) # Horizontal
    
                ax.set_xlabel('Count', color=COLOR) # Horizontal
                ax.set_yticks(label_locs) # Horizontal
                ax.set_yticklabels(labels)
    
                autolabel_horizontal(rects, ax, COLOR, column_width, bar_type='single')
                
            elif orientation=='vertical':             
                width = 1
                rects = ax.bar(label_locs, data, alpha=0.5, color=colorlist[0]) # Vertical
    
                ax.set_ylabel('Count', color=COLOR) # Vertical
                ax.set_xticks(label_locs) # Vertical
                ax.set_xticklabels(labels) # Vertical
                ax.set_xticklabels([wrap_label(label.get_text(), 15) for label in ax.get_xticklabels()])
    
                autolabel_vertical(rects, ax, COLOR, width, bar_type='single')
                
            plt.title(title, color=COLOR)

            plot_filename = "Images\\"+mode+"_"+orientation+"_barplot_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def stacked_bar(df, top_group, group, orientation=None, title=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        top_groups = df.groupby(top_group)[top_group].count()
    
        group_1 = top_groups.index[0]
        group_2 = top_groups.index[1]
        
        grouped = df.groupby([top_group, group])[group].count()
    
        group_1_grouped = grouped[group_1].sort_index(ascending=True)
        group_2_grouped = grouped[group_2].sort_index(ascending=True)

        results = {group_1: [],
                   group_2: []}

        category_list = list(df.groupby(group)[group].count().index)

        for category in category_list:
        
            try:
                num1 = group_1_grouped.loc[category]
            except:
                num1 = 0
        
            results[group_1].append(num1)
        
            try:
                num2 = group_2_grouped.loc[category]
            except:
                num2 = 0
    
            results[group_2].append(num2)
    
        labels = list(group_1_grouped.index)
        label_locs = np.arange(len(labels))  # the label locations
      
        fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed
        
        for mode, color in backgrounds.items():

            COLOR = color

            if orientation=='horizontal':
    
                column_width = 0.75
                # the width of the bars
                rects1 = ax.barh(labels,
                                 results[group_1],
                                 column_width,
                                 alpha=0.5,
                                 color=colorlist[3]) # Horizontal
                rects2 = ax.barh(labels,
                                 results[group_2],
                                 column_width,
                                 left=results[group_1],
                                 alpha=0.5,
                                 color=colorlist[2]) # Horizontal
    
                ax.set_xlabel('Count', color=COLOR) # Horizontal
                ax.set_yticks(label_locs) # Horizontal
                ax.set_yticklabels(labels)
    
                autolabel_horizontal(rects1, ax, COLOR, column_width, bar_type='stacked')
                autolabel_horizontal(rects2, ax, COLOR, column_width, bar_type='stacked')
                
            elif orientation=='vertical':

                width = 0.35
                rects1 = ax.bar(labels,
                                results[group_1],
                                width,
                                alpha=0.5,
                                color=colorlist[3]) # Vertical
                rects2 = ax.bar(labels,
                                results[group_2],
                                width,
                                bottom=results[group_1],
                                alpha=0.5,
                                color=colorlist[2]) # Vertical
    
                ax.set_ylabel('Count', color=COLOR) # Vertical
                ax.set_xticks(label_locs) # Vertical
                ax.set_xticklabels(labels) # Vertical
                ax.set_xticklabels([wrap_label(label.get_text(), 15) for label in ax.get_xticklabels()])
    
                autolabel_vertical(rects1, ax, COLOR, width, bar_type='stacked')
                autolabel_vertical(rects2, ax, COLOR, width, bar_type='stacked')

            plt.title(title, color=COLOR)
            
            ax.tick_params(axis='both', labelcolor=COLOR, colors=COLOR)

            ax.legend(labels=top_groups.index.to_list(),
                      bbox_to_anchor=(1, 0.975),
                      loc='upper left',
                      labelcolor=colorlist[0]).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)

            plot_filename = "Images\\"+mode+"_"+orientation+"_stacked_barplot_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def fitter(group, group_name):

        f = Fitter(group, distributions=get_common_distributions())
        f.fit()
        f.summary() # Prints a summary table and a plot
        filename = 'Images//fitter_'+group_name+'_'+col+'.png'
        plt.savefig(filename, bbox_inches='tight', dpi=300)
        
        best_fit = f.get_best()
        best_fit = list(best_fit.keys())[0]

        return best_fit
    
    def grouped_bar(df, top_group, group, orientation=None, title=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        top_groups = df.groupby(top_group)[top_group].count()
        
        group_1 = top_groups.index[0]
        group_2 = top_groups.index[1]
        
        grouped = df.groupby([top_group, group])[group].count()
        
        group_1_grouped = grouped[group_1].sort_index(ascending=True)
        group_2_grouped = grouped[group_2].sort_index(ascending=True)

        results = {group_1: [],
                   group_2: []}

        category_list = list(df.groupby(group)[group].count().index)

        for category in category_list:        
            try:
                num1 = group_1_grouped.loc[category]
            except:
                num1 = 0
        
            results[group_1].append(num1)
        
            try:
                num2 = group_2_grouped.loc[category]
            except:
                num2 = 0
    
            results[group_2].append(num2)
    
        labels = list(group_1_grouped.index)
        label_locs = np.arange(len(labels))  # the label locations

        for mode, color in backgrounds.items():

            COLOR = color

            fig, ax = plt.subplots(figsize=(9, 5))

            if orientation=='horizontal':
                column_width = 0.3
                # the width of the bars
                rects1 = ax.barh(label_locs - column_width/2,
                                 results[group_1],
                                 column_width,
                                 label=group_1,
                                 alpha=0.5,
                                 color=colorlist[0]) # Horizontal
                rects2 = ax.barh(label_locs + column_width/2,
                                 results[group_2],
                                 column_width,
                                 label=group_2,
                                 alpha=0.5,
                                 color=colorlist[3]) # Horizontal
    
                ax.set_xlabel('Count', color=COLOR) # Horizontal
                ax.set_yticks(label_locs) # Horizontal
                ax.set_yticklabels(labels)
    
                autolabel_horizontal(rects1, ax, COLOR, column_width, bar_type='grouped')
                autolabel_horizontal(rects2, ax, COLOR, column_width, bar_type='grouped')
    
            elif orientation=='vertical':
                width = 0.35
                
                rects1 = ax.bar(label_locs - width/2, results[group_1], width, label=group_1, color=colorlist[0], alpha=0.5) # Vertical
                
                rects2 = ax.bar(label_locs + width/2, results[group_2], width, label=group_2, color=colorlist[3], alpha=0.5) # Vertical
    
                ax.set_ylabel('Count', color=COLOR) # Vertical
                ax.set_xticks(label_locs) # Vertical
                ax.set_xticklabels(labels) # Vertical
                ax.set_xticklabels([wrap_label(label.get_text(), 15) for label in ax.get_xticklabels()])
    
                autolabel_vertical(rects1, ax, COLOR, width, bar_type='grouped')
                autolabel_vertical(rects2, ax, COLOR, width, bar_type='grouped')

            ax.tick_params(axis='both', labelcolor=COLOR, colors=COLOR)

            ax.legend()
            plt.title(title, color=COLOR)
            fig.tight_layout()
    
            plot_filename = "Images\\"+mode+"_"+orientation+"_grouped_barplot_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def histogram(df, col, x_label, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(6, 5)) # Adjust figure size as needed

            data = df[col]
            abs_z_scores  = np.abs(zscore(data))
            outliers = data[abs_z_scores > 3]
            filtered_data = data[abs_z_scores <= 3]
            
            if df[col].nunique()<10:
                bins = df[col].nunique()
            elif filtered_data.nunique() <=1000:
                bins = round(df[col].nunique()/50)
            else:
                bins = 50

            median = int(df[col].median())
            
            y, x, _ = plt.hist(filtered_data,
                     bins=bins,
                     color=colorlist[0])

            print(f'There are {len(outliers):,} values above {int(x.max())} days not shown in figure.')
  
            plt.fill([median, median, median, median],
                     [0, y.max(), y.max(), 0], color = 'black')
            median_annotation = "Median LoS: " + str(median) + " days"

            if median < 25:
                plt.text(median+1, y.max(), median_annotation, fontsize=14)
            else:
                plt.text(median+5, y.max()-y.max()/2, median_annotation, fontsize=14)
            
            plt.xlabel(x_label)            
            plt.ylabel("Frequency")
            
            title = col
            plt.title(title)
            
            ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
                            
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color
    
            plot_filename = "Images\\"+mode+"_histogram_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def overlapping_histograms(groups, group_names, col, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']
        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(10, 5)) # Adjust figure size as needed
            bins = groups[0].nunique()

            if bins<10:
                pass
            else:
                bins = 10

            inc = 0
            for x, group in enumerate(groups):
                plt.hist(group, color=colorlist[x+inc],
                         alpha=0.5,
                         bins=bins,
                         rwidth=0.9,
                         edgecolor = 'gray',
                         label=group_names[x])
                inc+=2

            plt.xlabel("Value")
            plt.ylabel("Frequency")
            plt.title(col)
            
            ax.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
            ax.legend()
                            
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color
    
            plot_filename = "Images\\"+mode+"_histogram_"+col.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)
    
    def histograms_subplots(groups, group_names, col, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']
        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')

            data = groups[0]
            abs_z_scores  = np.abs(zscore(data))
            outliers = data[abs_z_scores > 3]
            filtered_data_0 = data[abs_z_scores <= 3]
            
            if data.nunique()<=50:
                bins = data.nunique()
            else:
                bins = 50

            data = groups[1]
            abs_z_scores  = np.abs(zscore(data))
            outliers = data[abs_z_scores > 3]
            filtered_data_1 = data[abs_z_scores <= 3]

            ax1.hist(filtered_data_0, 
                     color=colorlist[0],
                     alpha=0.5,
                     bins=bins,
                     rwidth=0.9,
                     align = 'mid',
                     edgecolor = 'gray',
                     label=group_names[0])
            
            ax1.set_title('Fast Adoptions', fontsize=14)
            ax1.set_xlabel('Value', fontsize=12)
            ax1.set_ylabel('Frequency', fontsize=12)
            
            ax2.hist(filtered_data_1, 
                     color=colorlist[2],
                     alpha=0.5,
                     bins=bins,
                     rwidth=0.9,
                     align = 'mid',
                     edgecolor = 'gray',
                     label=group_names[1])
            ax2.set_title('Slow Adoptions', fontsize=14)
            ax2.set_xlabel('Value', fontsize=12)
            
            # Adjust layout and display the plot
            # plt.tight_layout()
            plt.suptitle(col, fontsize=18,
                         # y=-0.95
                        )
            
            ax1.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
            ax2.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))
                            
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color
    
            plot_filename = "Images\\"+mode+"_histogram_sub_"+col.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def word_cloud(df, animal, col, scheme):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        subset = df[df['species']==animal.title()]

        text_list = list(chain(subset[col]))
        text_list = [t for t in text_list if t!='unknown' and type(t)==type('string')]
        text_list = [t.replace(',',' ').replace('[',' ').replace(']',' ') for t in text_list]
        text_list = [t.replace("'"," ") for t in text_list]
        text_list = "".join(text_list)
        data = text_list.replace('|', ' ').strip()

        replace_words = ['web_address', 'dollar_amount', 'email_address', 'age_of_pet', 'test',
                         'weight_of_pet', 'physical_address', 'day_of_year', 'phone_number',
                         'day_of_week']

        for word in replace_words:
            data = data.replace(word, '')
        
        if animal=='dog':
            mask = np.array(Image.open("dog_portrait.jpg"))
        else:
            mask = np.array(Image.open("cat_portrait.jpg"))

        # Transform mask to be clearer or something
        mask[mask == 0] = 255 
        ## Things to do
        ## Change the font
        ## Fix the masks
        ## Reduce the blurriness
        ## Add a contour to the background - figured it out, looks terrible
        
        wc = WordCloud(
                       width=800,
                       height=400,
                       background_color=None, # default: black
                       max_words=100,
                       mask=mask,
                       stopwords=stopwords.words(),
                       colormap=colormap,
                       mode="RGBA",
                       repeat=False,
                       collocations=False
                    ).generate(data)

        wc.to_file("wordcloud.png") # Not 100% sure I need this
        plt.figure(figsize=(15,15)).gca()
        plt.imshow(wc, interpolation='bilinear')
        plt.axis('off')
        title = "Word Cloud of " + col.title() + " for " + animal.title()
        plt.tight_layout(pad=0)

        plot_filename = "Images\\"+"word_cloud_"+title.replace(" ", "_")+"_.png" 
        plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def model_comparison(cv_results, model_list, feature_set, title, comparison=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']

        for idx, x in enumerate(cv_results):
            if x[0]<1:
                cv_results[idx] = x*100
    
        num_model = len(cv_results)

        color = 'black'
        mode = 'light'
        fig, ax = plt.subplots(figsize=(num_model*2, 5))

        bplot = plt.boxplot(cv_results,
                            labels=model_list,
                            medianprops=dict(color=colorlist[0], linewidth=2), # Median line color
                            patch_artist=True, boxprops=dict(edgecolor=color, linewidth=1), # box outline
                            whiskerprops=dict(color=color, linewidth=1), # whiskers
                            capprops=dict(color=color), # whisker ends
                            flierprops=dict(markerfacecolor=colorlist[0], marker='o', markeredgecolor=colorlist[0]) # Colors outliers
                           )

        for median in bplot['medians']:
            # Get the x and y data of the median line
            x, y = median.get_xdata(), median.get_ydata()
            # Calculate the median value (it's the same value at both ends of the line)
            median_value = y[0]
            # Place the annotation text. Adjust the x position slightly for better placement.
            ax.text(np.average(x)+.25,
                    median_value,
                    str(round(median_value,2))+'%',
                    va='center',
                    ha='left',
                    fontsize=12
                   )

        # Fill the boxes with colors
        for patch in bplot['boxes']:
            patch.set_facecolor(colorlist[1])

        plt.plot(cmap=colormap) # You can choose different colormaps
        # ax.xaxis.label.set_color(color)
        plt.rcParams['text.color'] = color
        plt.rcParams['axes.labelcolor'] = color
        plt.rcParams['xtick.color'] = color
        plt.rcParams['ytick.color'] = color
            
        ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
        
        plt.title(title)
        plt.ylabel("Accuracy Scores")
        ax.set_xticklabels([wrap_label(label.get_text(), 15) for label in ax.get_xticklabels()])
    
        plot_filename = "Images\\"+mode+"_model_comparison_"+title.replace(" ", "_")+".png" 
        plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def confusion_matrix(confusion_matrix, version, scheme=None):
    
        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']
        
        for mode, color in backgrounds.items():

            plt.rcParams['axes.grid'] = False
            
            ## Check these labels
            display_labels = ['Fast Adoption', 'Slow Adoption']
        
            disp = ConfusionMatrixDisplay(confusion_matrix, display_labels=display_labels)
        
            disp.plot(cmap=colormap) # You can choose different colormaps
            disp.ax_.xaxis.label.set_color(color)
            disp.ax_.yaxis.label.set_color(color)
            disp.ax_.tick_params(axis='x', colors=color) # Change x-axis tick label color
            disp.ax_.tick_params(axis='y', colors=color)
            title = 'Confusion Matrix'
            plt.title(title)
            
            plot_filename = "Images\\"+mode+"_confusion_matrix_"+version+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)
    
    def feature_importance(feature_importance_series, top=True, scheme=None):
    
        if top==True:
            feature_importance_series = feature_importance_series.sort_values(ascending=False)
            title = 'Feature Importance - Top 10 Features'
        else:
            feature_importance_series = feature_importance_series.sort_values(ascending=True)
            title = 'Feature Importance - Bottom 10 Features'
   
        top_ten = feature_importance_series[0:10]

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']
        
        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(9, 5)) # Adjust figure size as needed

            if top==True:
                plt.barh(top_ten.index[::-1], top_ten.values[::-1], color=colorlist[4])
            else:
                plt.barh(top_ten.index, top_ten.values, color=colorlist[0])
            
            plt.xlabel('Contribution (%) to model')
            plt.title(title)
    
            plot_filename = "Images\\"+mode+"feature_importance_"+title.replace(" ", "_")+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def roc_curve(y_test, y_pred_probs, auc, version, scheme=None):
    
        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']
        
        for mode, color in backgrounds.items():
    
            fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs, pos_label='slow')
            
            fig, ax = plt.subplots(figsize=(6, 5)) # Adjust figure size as needed
            plt.plot([0, 1], [0, 1], 'k--')

            annotation = np.round(auc, 2)
            plt.text(0.2, auc, annotation, color=color)
            
            # Plot tpr against fpr
            plt.plot(fpr, tpr)
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            title = 'ROC Curve for Adoption Type Prediction'
            plt.title(title)
        
            plot_filename = "Images\\"+mode+"_roc_curve_"+version+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def table(df, title, scheme=None):

        if len(df)>10:
            df = df[0:10]
        
        cols = df.columns
    
        for col in cols:
            # Make string if not already
            df[col] = df[col].apply(lambda x: round(x,4) if type(x)!=type('string') and type(x)!=type(None) else x)
            df[col] = df[col].apply(lambda x: str(x).capitalize())
            df[col] = df[col].apply(lambda x: x.replace('\n', '').replace('_', ' '))
            df[col] = df[col].apply(lambda x: x[0:15])

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']
        
        for mode, color in backgrounds.items():

            # Create a figure and an axes object
            fig, ax = plt.subplots(figsize=(10, 3))
            fig.set_facecolor('none')
            ax.set_facecolor('none')
            # Hide the axes for a cleaner table display
            ax.axis('off')
            ax.axis('tight')

            # Create the table
            table = ax.table(cellText=df.values,
                             colLabels=df.columns,
                             loc='center',
                             cellLoc='center')
    
            # Adjust cell properties for better aesthetics
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1.3, 1.3) # Scale for padding
    
            for cell in table._cells:
                table._cells[cell].set_alpha(0.0) # 0.0 for fully transparent background
                table._cells[cell].set_facecolor(None)
                table._cells[cell].set_edgecolor('white')
                table._cells[cell].set_linewidth(1.0)
        
            plt.title(title)
            
            plot_filename = "Images\\"+mode+"table_"+title.replace(" ", "_")+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def learning_curve(train_sizes, train_scores, test_scores, goal_accuracy=None, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']

        backgrounds = plot_color_scheme['backgrounds']

        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(8, 6))
            
            train_scores_mean = np.mean(train_scores, axis=1)*100
            train_scores_std = np.std(train_scores, axis=1)*100
            test_scores_mean = np.mean(test_scores, axis=1)*100
            test_scores_std = np.std(test_scores, axis=1)*100

            plt.plot(train_sizes, train_scores_mean, label="Training Accuracy", color=colorlist[2])
            plt.plot(train_sizes, test_scores_mean, label="Testing Accuracy", color=colorlist[0])
            
            plt.fill_between(train_sizes,
                             train_scores_mean - train_scores_std,
                             train_scores_mean + train_scores_std,
                             alpha=0.2, color=colorlist[2])
            plt.fill_between(train_sizes,
                             test_scores_mean - test_scores_std,
                             test_scores_mean + test_scores_std,
                             alpha=0.2, color=colorlist[0])

            if goal_accuracy:
                # Test out a linear fit line
                popt, pcov = curve_fit(linear_func, train_sizes, test_scores_mean)
                a_opt, b_opt = popt
                x_model = np.linspace(min(train_sizes), max(train_sizes), 100)
                y_model = linear_func(x_model, a_opt, b_opt)
                plt.plot(x_model, y_model, color='black', label=f'Linear fit: y={a_opt:.5f}x + {b_opt:.5f}')
                print(f'It will take approximately {round((goal_accuracy-b_opt)/a_opt):,} samples to reach {goal_accuracy}% accuracy')
                
            # # Test out a log fit line
            # popt, pcov = curve_fit(log_func, train_sizes, test_scores_mean)
            # a_opt, b_opt = popt
            # x_model = np.linspace(min(train_sizes), max(train_sizes), 100)
            # y_model = log_func(x_model, a_opt, b_opt)
            # plt.plot(x_model, y_model, color='gray', label=f'Log fit: y={a_opt:.5f}*log(x) + {b_opt:.5f}')
            # print(f'It will take approximately {round(np.exp((goal_accuracy-b_opt)/a_opt)):,} samples to reach {goal_accuracy}% accuracy')
            
            plt.xlabel("Number of samples in training set", color=color)
            plt.ylabel("Score", color=color)
            plt.title('Learning Curve', color=color)
            
            ax.tick_params(axis='x', colors=color)
            ax.tick_params(axis='y', colors=color) 

            ax.legend(bbox_to_anchor=(1, 0.975),
                      loc='upper left',
                      labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
            ax.xaxis.set_major_formatter(FuncFormatter(thousands_formatter))
    
            plot_filename = "Images\\"+mode+"_learning_curve"+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

        return train_sizes, test_scores_mean

    def model_complexity(data, param_name, scheme=None):

        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}
    
        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']

        backgrounds = plot_color_scheme['backgrounds']
    
        for mode, color in backgrounds.items():
            
            fig, ax = plt.subplots(figsize=(8, 6))
    
            plt.plot(data['hyperparameter_values'], data['train_means'], label="Training Accuracy", color=colorlist[2])
            plt.plot(data['hyperparameter_values'], data['test_means'],  label="Testing Accuracy", color=colorlist[0])
        
            plt.fill_between(data['hyperparameter_values'],
                             (data['train_means'] - data['train_stds']),
                             (data['train_means'] + data['train_stds']),
                             alpha=0.2, color=colorlist[3])
            plt.fill_between(data['hyperparameter_values'],
                             (data['test_means'] - data['test_stds']),
                             (data['test_means'] + data['test_stds']),
                             alpha=0.2, color=colorlist[1])
            
            index = np.argmax(data['test_means'])
            increment = data['hyperparameter_values'][1]-data['hyperparameter_values'][1]
            plt.text(data['hyperparameter_values'][index], data['test_means'].max(), "*", color=color)
            plt.text(data['hyperparameter_values'][index]-(increment),
                     data['test_means'].max()+.2,
                     param_name + ": " + str(data['hyperparameter_values'][index]), color=color)

            title = 'Model complexity curve for ' + param_name
            plt.title(title, color=color)
            plt.xlabel(param_name + " values", color=color)
            plt.ylabel("Score", color=color)

            ax.tick_params(axis='x', colors=color)
            ax.tick_params(axis='y', colors=color)            
            ax.legend(bbox_to_anchor=(1, 0.975),
                      loc='upper left',
                      labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
        
            plot_filename = "Images\\"+mode+"_model_complexity_curve_"+param_name+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

        return data['hyperparameter_values'][index]

    def feature_complexity(data, num_features, scheme=None, order=None):
        
        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}
        
        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']
        
        backgrounds = plot_color_scheme['backgrounds']
    
        num_vals = len(data['test_means'])
        num_features = num_features+1
        num_features = list(range(num_features-num_vals, num_features, 1))
        num_features = num_features[::-1]
    
        for mode, color in backgrounds.items():
    
            fig, ax = plt.subplots(figsize=(8, 6))
        
            plt.plot(num_features, np.array(data['test_means']), label="Testing Accuracy", color=colorlist[0])
            plt.plot(num_features, np.array(data['train_means']), label="Training Accuracy", color=colorlist[3])
        
            plt.fill_between(num_features,
                             np.array(data['test_means']) - np.array(data['test_stds']),
                             np.array(data['test_means']) + np.array(data['test_stds']),
                             alpha=0.2,
                             color=colorlist[0]
                            )
        
            plt.fill_between(num_features,
                             np.array(data['train_means']) - np.array(data['train_stds']),
                             np.array(data['train_means']) + np.array(data['train_stds']),
                             alpha=0.2,
                             color=colorlist[3]
                            )
        
            max_acc = np.max(np.array(data['test_means']))
            
            if order=='backward':
                ax.invert_xaxis()
                indices = np.where(data['test_means'][::-1] == max_acc)[0][0]
                x_pos = indices+5
            else:
                indices = np.where(data['test_means'] == max_acc)[0][0]
                x_pos = indices-5
                
            annotation = f'Max accuracy ({max_acc:.4}%) achieved with {indices} features.'
            plt.text(x_pos, max_acc+.5, annotation, fontsize=14)
            plt.text(indices, max_acc, '*', fontsize=14)
            
            plt.xlabel("# Features")
            plt.ylabel("Accuracy Score")
            ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
            plt.title('Feature Complexity Curve')
    
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color
        
            ax.legend(bbox_to_anchor=(1, 0.975),
                          loc='upper left',
                          labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            ax.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
        
            plot_filename = "Images\\"+mode+"feature_complexity_curve_"+"_.png" 
            plt.savefig(plot_filename, bbox_inches = 'tight', transparent=True)

    def feature_accuracy(features, accuracies, order=None, scheme=None):
        
        colors = {'bpn': color_scheme.bpn_colors(),
                  'pmi': color_scheme.pmi_colors()}

        plot_color_scheme = colors[scheme]
        colorlist = plot_color_scheme['colorlist']
        colormap = plot_color_scheme['colormap']

        backgrounds = plot_color_scheme['backgrounds']
        
        for mode, color in backgrounds.items():

            fig, ax = plt.subplots(figsize=(10, 16)) # Adjust figure size as needed
            patch1 = mpatches.Patch(color=colorlist[4], label='Remove')
            patch2 = mpatches.Patch(color=colorlist[3], label='Max Accuracy')
            patch3 = mpatches.Patch(color=colorlist[1], label='Keep')
            max_acc = np.max(accuracies)

            if order=='backward':
                indices = np.where(accuracies[::-1] == max_acc)[0][0]
                clrs = [colorlist[1] if x<indices else 
                        colorlist[3] if x==indices else
                        colorlist[4] for x,i in enumerate(dropped_features[::-1])]
                handles = [patch3 if x<indices else
                           patch2 if x==indices else
                           patch1 for x,i in enumerate(dropped_features[::-1])]
                plt.barh(dropped_features[::-1], accuracies[::-1], color=clrs, alpha=0.8)
                title = 'Backward Feature Elimination'
            else:
                indices = np.where(accuracies == max_acc)[0][0]
                clrs = [colorlist[4] if x<indices else 
                        colorlist[3] if x==indices else
                        colorlist[1] for x,i in enumerate(dropped_features)]
                handles = [patch1 if x<indices else
                           patch2 if x==indices else
                           patch3 for x,i in enumerate(dropped_features)]
                plt.barh(dropped_features, accuracies, color=clrs, alpha=0.8)
                title = 'Forward Feature Selection'

            annotation = 'Max accuracy'

            plt.xlim(int(math.floor(np.min(accuracies))), int(math.ceil(np.max(accuracies))))
            ax.xaxis.set_major_formatter(FuncFormatter(percentage_formatter))

            ax.legend(handles=list(set(handles)),
                      bbox_to_anchor=(1, 0.975),
                      labelcolor=backgrounds['light']).get_frame().set_boxstyle('Round', pad=0.1, rounding_size=1)
            
            plt.rcParams['text.color'] = color
            plt.rcParams['axes.labelcolor'] = color
            plt.rcParams['xtick.color'] = color
            plt.rcParams['ytick.color'] = color
            plt.title(f'Model Accuracies - {title}')
            plt.xlabel('Accuracy')
            plt.ylabel('Removed Feature')
            
            plot_filename = "Images\\" + mode + "_model_accuracies_"+title+".png" 
            plt.savefig(plot_filename, bbox_inches = 'tight',
                        # facecolor='white', 
                        transparent=True)
            plt.show()

In [ ]:
# Define a custom formatting function
def percentage_formatter(x, pos):
    return f'{x/100:.0%}'

def autolabel_vertical(rects, axes, color, width, bar_type=None):
    """Attach a text label above each bar in *rects*, displaying its height."""
    for rect in rects:
        height = rect.get_height()

        if bar_type=='grouped':
            x_loc = rect.get_x() + rect.get_width()-(width/2)
            y_loc = height+0.15
        elif bar_type=='stacked':
            x_loc = rect.get_x() + rect.get_width()-(width/2)            
            y_loc = height*.45
            color = 'black'        
        elif bar_type=='single':
            x_loc = rect.get_x() + rect.get_width()-(width*.4)
            y_loc = height+0.5
            
        if rect.get_height()> 1:
            axes.annotate('{}'.format(height),
                           xy=(x_loc, y_loc),
                           textcoords="data",
                           ha='center',
                           va='bottom',
                           color=color)

def autolabel_horizontal(rects, axes, color, col_width, bar_type=None):
    for rect in rects:
        width = rect.get_width() # This is the actual number that gets plotted
        
        if bar_type=='grouped':
            height = (rect.get_y()+rect.get_height()-col_width)+0.015
            x_loc = width+0.1
        elif bar_type=='stacked':
            height = (rect.get_y()+rect.get_height()-(col_width*.65))
            x_loc = width*.5
            color = 'black'
        elif bar_type=='single':
            height = (rect.get_y()+rect.get_height()-(col_width*.55))+0.015
            x_loc = width+.5
            
        if rect.get_width()> 1:
            axes.annotate('{}'.format(width),
                           xy=(x_loc, height),
                           textcoords="data",
                           va='bottom',
                           ha='center',
                           color=color)

def wrap_label(label, width):
    return '\n'.join(textwrap.wrap(label, width))

def thousands_formatter(x, pos):
    return f'{int(x):,}'

def to_percent_formatter(x, pos):
    x = int(round(x*100,0))
    num = str(x)+'%'
    return f'{num}'

def log_func(x, a, b):
    """Log function: y = a*log(x) + b"""
    return a * np.log(x) + b

def linear_func(x, a, b):
    """Linear function: y = a*x + b"""
    return a * x + b